In [18]:
# -*- coding: utf-8 -*-

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "colab"


class ZbusImpedanceSweep:
    def __init__(self) -> None:
        self.id_to_idx: dict[int, int] = {}
        self.idx_to_id: dict[int, int] = {}

    @staticmethod
    def z_series_RLC(w: float, R: float = 0.0, L: float = 0.0, C: float = 0.0) -> complex:
        Z = complex(R, 0.0)

        if L != 0.0:
            Z = Z + 1j * w * L

        if C != 0.0:
            Z = Z + 1.0 / (1j * w * C)

        return Z

    @staticmethod
    def zbuild_from_Zbranch(branch_data: np.ndarray) -> np.ndarray:
        bd = np.array(branch_data, dtype=object)
        nl = bd[:, 0].astype(int)
        nr = bd[:, 1].astype(int)
        ZB = np.array(bd[:, 2], dtype=complex)

        nbr = int(bd.shape[0])
        nbus = int(max(np.max(nl), np.max(nr)))

        Zbus = np.zeros((nbus, nbus), dtype=complex)
        ntree = np.ones(nbr, dtype=int)
        tree = 0

        for i in range(nbr):
            if (nl[i] == 0) or (nr[i] == 0):
                n = nr[i] if nl[i] == 0 else nl[i]
                n_idx = n - 1

                if np.abs(Zbus[n_idx, n_idx]) == 0.0:
                    Zbus[n_idx, n_idx] = ZB[i]
                    tree = tree + 1
                else:
                    Zbus[n_idx, n_idx] = (Zbus[n_idx, n_idx] * ZB[i]) / (Zbus[n_idx, n_idx] + ZB[i])

                ntree[i] = 2

        while tree < nbus:
            for n in range(1, nbus + 1):
                n_idx = n - 1

                if np.abs(Zbus[n_idx, n_idx]) == 0.0:
                    for i in range(nbr):
                        if (nl[i] == n) or (nr[i] == n):
                            k_bus = nr[i] if nl[i] == n else nl[i]
                            if k_bus == 0:
                                continue

                            k_idx = k_bus - 1

                            if np.abs(Zbus[k_idx, k_idx]) != 0.0:
                                for m in range(1, nbus + 1):
                                    m_idx = m - 1
                                    if m != n:
                                        Zbus[m_idx, n_idx] = Zbus[m_idx, k_idx]
                                        Zbus[n_idx, m_idx] = Zbus[m_idx, k_idx]

                                Zbus[n_idx, n_idx] = Zbus[k_idx, k_idx] + ZB[i]
                                tree = tree + 1
                                ntree[i] = 2
                                break

        for i in range(nbr):
            if ntree[i] == 1:
                n_idx = nl[i] - 1
                k_idx = nr[i] - 1

                DM = Zbus[n_idx, n_idx] + Zbus[k_idx, k_idx] + ZB[i] - 2.0 * Zbus[n_idx, k_idx]

                if np.abs(DM) != 0.0:
                    DELZ = np.zeros((nbus, nbus), dtype=complex)

                    for jj in range(nbus):
                        AP = Zbus[jj, n_idx] - Zbus[jj, k_idx]
                        for kk in range(nbus):
                            AT = Zbus[n_idx, kk] - Zbus[k_idx, kk]
                            DELZ[jj, kk] = (AP * AT) / DM

                    Zbus = Zbus - DELZ
                    ntree[i] = 2

        return Zbus

    @staticmethod
    def validate_inputs(series_branches: np.ndarray, shunt_RLC: np.ndarray) -> None:
        if series_branches.ndim != 2 or series_branches.shape[1] != 5:
            raise ValueError("series_branches must have 5 columns: [from, to, R, L, C].")

        if shunt_RLC.ndim != 2 or shunt_RLC.shape[1] != 4:
            raise ValueError("shunt_RLC must have 4 columns: [bus, R, L, C].")

        for row in series_branches:
            a = float(row[0])
            b = float(row[1])
            if abs(a - round(a)) > 0.0 or abs(b - round(b)) > 0.0:
                raise ValueError("series_branches bus IDs must be integers (0 allowed).")

        for row in shunt_RLC:
            a = float(row[0])
            if abs(a - round(a)) > 0.0:
                raise ValueError("shunt_RLC bus IDs must be integers.")

    def remap_bus_ids(self, series_branches: np.ndarray, shunt_RLC: np.ndarray) -> tuple:
        ids = set()

        for row in series_branches:
            a = int(row[0])
            b = int(row[1])
            if a != 0:
                ids.add(a)
            if b != 0:
                ids.add(b)

        for row in shunt_RLC:
            bus_id = int(row[0])
            if bus_id != 0:
                ids.add(bus_id)

        if len(ids) == 0:
            raise ValueError("No non-zero buses found.")

        ids_sorted = sorted(list(ids))

        self.id_to_idx = {}
        self.idx_to_id = {}

        internal_idx = 1
        for bus_id in ids_sorted:
            self.id_to_idx[int(bus_id)] = int(internal_idx)
            self.idx_to_id[int(internal_idx)] = int(bus_id)
            internal_idx = internal_idx + 1

        series_mapped_list = []
        for row in series_branches:
            a = int(row[0])
            b = int(row[1])

            if a != 0:
                a = self.id_to_idx[a]
            if b != 0:
                b = self.id_to_idx[b]

            series_mapped_list.append([a, b, float(row[2]), float(row[3]), float(row[4])])

        series_mapped = np.array(series_mapped_list, dtype=float)

        shunt_mapped_list = []
        for row in shunt_RLC:
            bus_id = int(row[0])
            if bus_id != 0:
                bus_id = self.id_to_idx[bus_id]
            shunt_mapped_list.append([bus_id, float(row[1]), float(row[2]), float(row[3])])

        shunt_mapped = np.array(shunt_mapped_list, dtype=float)

        bus_ids_list = []
        i = 1
        while i <= len(self.idx_to_id):
            bus_ids_list.append(int(self.idx_to_id[i]))
            i = i + 1

        bus_ids = np.array(bus_ids_list, dtype=int)
        return series_mapped, shunt_mapped, bus_ids, self.id_to_idx, self.idx_to_id

    def sweep_zbus_diagonal(self, series_branches: np.ndarray, shunt_RLC: np.ndarray, f_hz: np.ndarray) -> tuple:
        self.validate_inputs(series_branches, shunt_RLC)

        series_m, shunt_m, bus_ids, _, _ = self.remap_bus_ids(series_branches, shunt_RLC)

        nf = int(f_hz.size)
        nbus = int(bus_ids.size)

        Zeq_diag = np.zeros((nf, nbus), dtype=complex)

        k = 0
        while k < nf:
            w = 2.0 * np.pi * f_hz[k]
            branch_list = []

            for row in series_m:
                z = self.z_series_RLC(w, R=row[2], L=row[3], C=row[4])
                branch_list.append([int(row[0]), int(row[1]), z])

            for row in shunt_m:
                z_sh = self.z_series_RLC(w, R=row[1], L=row[2], C=row[3])
                branch_list.append([0, int(row[0]), z_sh])

            Zbus_f = self.zbuild_from_Zbranch(np.array(branch_list, dtype=object))
            Zeq_diag[k, :] = np.diag(Zbus_f)

            k = k + 1

        return f_hz, Zeq_diag, bus_ids

    @staticmethod
    def make_plotly_magnitude_figure(f_hz: np.ndarray, Zeq_diag: np.ndarray, bus_ids: np.ndarray) -> go.Figure:
        fig = go.Figure()
        eps = 1.0e-12

        i = 0
        while i < Zeq_diag.shape[1]:
            y = np.abs(Zeq_diag[:, i])
            y = np.maximum(y, eps)

            fig.add_trace(go.Scatter(
                x=f_hz,
                y=y,
                mode="lines",
                name="Bus ID {}".format(int(bus_ids[i])),
                connectgaps=True
            ))
            i = i + 1

        x_min = float(np.min(f_hz))
        x_max = float(np.max(f_hz))

        fig.update_xaxes(type="log", title_text="Frequency (Hz)", autorange=False, range=[np.log10(x_min), np.log10(x_max)])
        fig.update_yaxes(type="log", title_text="|Zii(f)| (Ohm)")
        fig.update_layout(title="Driving-Point Impedance Magnitude", hovermode="x unified")
        return fig

    @staticmethod
    def make_plotly_phase_figure(f_hz: np.ndarray, Zeq_diag: np.ndarray, bus_ids: np.ndarray) -> go.Figure:
        fig = go.Figure()

        i = 0
        while i < Zeq_diag.shape[1]:
            phase_deg = np.angle(Zeq_diag[:, i], deg=True)

            fig.add_trace(go.Scatter(
                x=f_hz,
                y=phase_deg,
                mode="lines",
                name="Bus ID {}".format(int(bus_ids[i])),
                connectgaps=True
            ))
            i = i + 1

        x_min = float(np.min(f_hz))
        x_max = float(np.max(f_hz))

        fig.update_xaxes(type="log", title_text="Frequency (Hz)", autorange=False, range=[np.log10(x_min), np.log10(x_max)])
        fig.update_yaxes(title_text="Phase(Zii) (deg)")
        fig.update_layout(title="Driving-Point Impedance Phase", hovermode="x unified")
        return fig

    @staticmethod
    def make_plotly_normalized_magnitude_figure(
        f_hz: np.ndarray,
        Zeq_diag: np.ndarray,
        bus_ids: np.ndarray,
        f_ref_hz: float,
        eps_mag: float = 1.0e-12
    ) -> tuple:
        idx_ref, f_at_idx, Z_ref_vec = ZbusImpedanceSweep.get_index_frequency_and_impedances_at_frequency(
            f_hz=f_hz,
            Zeq_diag=Zeq_diag,
            bus_ids=bus_ids,
            f_target_hz=f_ref_hz
        )

        mag = np.abs(Zeq_diag)
        mag = np.maximum(mag, eps_mag)

        mag_ref = np.abs(Z_ref_vec)
        mag_ref = np.maximum(mag_ref, eps_mag)

        fig = go.Figure()

        i = 0
        while i < Zeq_diag.shape[1]:
            y = mag[:, i] / mag_ref[i]

            fig.add_trace(go.Scatter(
                x=f_hz,
                y=y,
                mode="lines",
                name="Bus ID {}".format(int(bus_ids[i])),
                connectgaps=True
            ))
            i = i + 1

        x_min = float(np.min(f_hz))
        x_max = float(np.max(f_hz))

        fig.update_xaxes(type="log", title_text="Frequency (Hz)", autorange=False, range=[np.log10(x_min), np.log10(x_max)])
        fig.update_yaxes(type="log", title_text="|Zii(f)| / |Zii({} Hz)|".format(float(f_at_idx)))
        fig.update_layout(title="Normalized Driving-Point Impedance Magnitude", hovermode="x unified")
        return fig, idx_ref, f_at_idx, Z_ref_vec

    @staticmethod
    def get_index_frequency_and_impedances_at_frequency(
        f_hz: np.ndarray,
        Zeq_diag: np.ndarray,
        bus_ids: np.ndarray,
        f_target_hz: float
    ) -> tuple:
        if f_hz.ndim != 1:
            raise ValueError("f_hz must be 1D.")
        if Zeq_diag.ndim != 2:
            raise ValueError("Zeq_diag must be 2D (nf x nbus).")
        if f_hz.size != Zeq_diag.shape[0]:
            raise ValueError("f_hz length must match Zeq_diag rows.")
        if bus_ids.size != Zeq_diag.shape[1]:
            raise ValueError("bus_ids length must match Zeq_diag cols.")

        idx_target = int(np.argmin(np.abs(f_hz - float(f_target_hz))))
        f_at_idx = float(f_hz[idx_target])
        Z_at_idx = np.array(Zeq_diag[idx_target, :], dtype=complex)

        return idx_target, f_at_idx, Z_at_idx

    @staticmethod
    def export_impedances_at_frequency_to_excel(
        filepath_xlsx: str,
        f_hz: np.ndarray,
        Zeq_diag: np.ndarray,
        bus_ids: np.ndarray,
        f_target_hz: float
    ) -> tuple:
        idx_target, f_at_idx, Z_at_idx = ZbusImpedanceSweep.get_index_frequency_and_impedances_at_frequency(
            f_hz=f_hz,
            Zeq_diag=Zeq_diag,
            bus_ids=bus_ids,
            f_target_hz=f_target_hz
        )

        try:
            import pandas as pd
        except ImportError as exc:
            raise ImportError("Install: pip install pandas openpyxl") from exc

        Z_real = np.real(Z_at_idx)
        Z_imag = np.imag(Z_at_idx)
        Z_mag = np.abs(Z_at_idx)
        Z_phase = np.angle(Z_at_idx, deg=True)

        bus_list = []
        f_list = []
        real_list = []
        imag_list = []
        mag_list = []
        phase_list = []

        i = 0
        while i < bus_ids.size:
            bus_list.append(int(bus_ids[i]))
            f_list.append(float(f_at_idx))
            real_list.append(float(Z_real[i]))
            imag_list.append(float(Z_imag[i]))
            mag_list.append(float(Z_mag[i]))
            phase_list.append(float(Z_phase[i]))
            i = i + 1

        df = pd.DataFrame({
            "bus_id": bus_list,
            "f_hz": f_list,
            "Z_real_ohm": real_list,
            "Z_imag_ohm": imag_list,
            "Z_mag_ohm": mag_list,
            "Z_phase_deg": phase_list,
        })

        df.to_excel(filepath_xlsx, index=False)
        return idx_target, f_at_idx, Z_at_idx

    @staticmethod
    def extract_harmonic_impedances(
        f_hz: np.ndarray,
        Zeq_diag: np.ndarray,
        bus_ids: np.ndarray,
        f_fund_hz: float,
        i_harmonics: np.ndarray
    ) -> tuple:
        if f_fund_hz <= 0.0:
            raise ValueError("f_fund_hz must be > 0.")

        f_min = float(np.min(f_hz))
        f_max = float(np.max(f_hz))

        h_min = int(np.ceil(f_min / f_fund_hz))
        if h_min < 1:
            h_min = 1

        h_max = int(np.floor(f_max / f_fund_hz))
        if h_max < 1:
            raise ValueError("Frequency range does not include any harmonic.")

        harmonics_list = []
        f_targets_list = []
        idx_list = []

        h = h_min
        while h <= h_max:
            f_t = float(h) * float(f_fund_hz)
            idx_t = int(np.argmin(np.abs(f_hz - f_t)))

            harmonics_list.append(int(h))
            f_targets_list.append(float(f_t))
            idx_list.append(int(idx_t))

            h = h + 1

        h_arr = np.array(harmonics_list, dtype=int)
        f_targets = np.array(f_targets_list, dtype=float)
        idx_arr = np.array(idx_list, dtype=int)

        nh = int(h_arr.size)
        nbus = int(bus_ids.size)

        Z_h = np.zeros((nh, nbus), dtype=complex)

        i = 0
        while i < nh:
            Z_h[i, :] = Zeq_diag[idx_arr[i], :]
            i = i + 1

        i_in = np.array(i_harmonics, dtype=complex)
        i_harmonics_out = np.zeros(nh, dtype=complex)

        n_copy = int(min(i_in.size, i_harmonics_out.size))
        if n_copy > 0:
            i_harmonics_out[:n_copy] = i_in[:n_copy]

        v_harmonics_out = i_harmonics_out.reshape(nh, 1) * Z_h
        return h_arr, f_targets, idx_arr, Z_h, i_harmonics_out, v_harmonics_out

    @staticmethod
    def make_plotly_harmonic_bar_current(
        h_arr: np.ndarray,
        i_harmonics_out: np.ndarray
    ) -> go.Figure:
        fig = go.Figure()

        y = np.abs(i_harmonics_out)

        fig.add_trace(go.Bar(
            x=h_arr,
            y=y,
            name="|I_h|"
        ))

        fig.update_xaxes(title_text="Harmonic order h (f = h * f_fund)")
        fig.update_yaxes(title_text="|I_h| (A)")
        fig.update_layout(
            title="Harmonic Currents (Magnitude)",
            hovermode="x unified"
        )
        return fig

    @staticmethod
    def make_plotly_harmonic_bar_voltage(
        h_arr: np.ndarray,
        v_harmonics_out: np.ndarray,
        bus_ids: np.ndarray
    ) -> go.Figure:
        fig = go.Figure()

        i = 0
        while i < bus_ids.size:
            y = np.abs(v_harmonics_out[:, i])

            fig.add_trace(go.Bar(
                x=h_arr,
                y=y,
                name="Bus ID {}".format(int(bus_ids[i]))
            ))
            i = i + 1

        fig.update_xaxes(title_text="Harmonic order h (f = h * f_fund)")
        fig.update_yaxes(title_text="|V_h| (V)")
        fig.update_layout(
            title="Harmonic Voltages per Bus (Magnitude)",
            barmode="group",
            hovermode="x unified"
        )
        return fig


def main() -> None:
    f_fund_hz = 60.0

    i_harmonics = np.array([100, 2, 30, 2, 20, 2, 10, 2, 5, 2], dtype=float)

    R_filter = 1.27e-3
    L_filter = 34.3e-3
    C_filter = 8.54e-6

    R_trafo = 23.8e-3
    L_trafo = 3.21e-3

    R_linha = 0.0
    L_linha = 48.4 * (34.5 / 220.0) ** 2 / 377.0

    series_branches = np.array([
        [0, 1, 9e99, 0.0, 0.0],
        [1, 2, R_trafo, L_trafo, 0.0],
        [2, 3, R_linha, L_linha, 0.0],
        [3, 0, R_trafo / 10.0, L_trafo / 10.0, 0.0],
    ], dtype=float)

    shunt_RLC = np.array([
        [1, R_filter, L_filter, C_filter],
    ], dtype=float)

    f_min_hz = f_fund_hz
    f_max_hz = 3000.0
    delta_freq = 1.0
    f_hz = np.arange(f_min_hz, f_max_hz + delta_freq, delta_freq, dtype=float)

    solver = ZbusImpedanceSweep()

    f_out, Zeq_diag, bus_ids = solver.sweep_zbus_diagonal(series_branches, shunt_RLC, f_hz)

    fig_mag = solver.make_plotly_magnitude_figure(f_out, Zeq_diag, bus_ids)
    fig_mag.show()

    fig_phase = solver.make_plotly_phase_figure(f_out, Zeq_diag, bus_ids)
    fig_phase.show()

    fig_norm, _, _, _ = solver.make_plotly_normalized_magnitude_figure(
        f_hz=f_out,
        Zeq_diag=Zeq_diag,
        bus_ids=bus_ids,
        f_ref_hz=f_fund_hz
    )
    fig_norm.show()

    h_arr, f_targets, idx_arr, Z_h, i_harmonics_out, v_harmonics_out = solver.extract_harmonic_impedances(
        f_hz=f_out,
        Zeq_diag=Zeq_diag,
        bus_ids=bus_ids,
        f_fund_hz=f_fund_hz,
        i_harmonics=i_harmonics
    )


    fig_i = solver.make_plotly_harmonic_bar_current(h_arr=h_arr, i_harmonics_out=i_harmonics_out)
    fig_i.show()

    fig_v = solver.make_plotly_harmonic_bar_voltage(h_arr=h_arr, v_harmonics_out=v_harmonics_out, bus_ids=bus_ids)
    fig_v.show()

    filepath_xlsx = r"zbus_impedances_at_fundamental.xlsx"
    solver.export_impedances_at_frequency_to_excel(
        filepath_xlsx=filepath_xlsx,
        f_hz=f_out,
        Zeq_diag=Zeq_diag,
        bus_ids=bus_ids,
        f_target_hz=f_fund_hz
    )
    print("")
    print("Excel exported to:", filepath_xlsx)

    print("")
    print("Harmonic vectors sizes:")
    print("len(h_arr) =", int(h_arr.size))
    print("len(f_targets) =", int(f_targets.size))
    print("len(i_harmonics_out) =", int(i_harmonics_out.size))
    print("v_harmonics_out shape =", v_harmonics_out.shape)


if __name__ == "__main__":
    main()



Excel exported to: zbus_impedances_at_fundamental.xlsx

Harmonic vectors sizes:
len(h_arr) = 50
len(f_targets) = 50
len(i_harmonics_out) = 50
v_harmonics_out shape = (50, 3)
